# 53 — Distributed Scheduling

**Engineering statement:** Distributed scheduling specifies coordinated adaptive serving.

Notebook 49 made one adaptive serving infrastructure loop deployable. Notebook 53 asks how many adaptive loops coordinate across routers, queues, replica pools, nodes, and regions.

```text
local adaptive infrastructure
→ shared telemetry
→ distributed queues
→ replica placement
→ routing coordination
→ load migration
→ coordinated serving
```

## Start here

Notebook 49 answered:

> How does one controller become adaptive serving infrastructure?

Notebook 53 asks:

> How do many controllers coordinate without turning coordination itself into the bottleneck?

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "53_distributed_scheduling"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 15,
    "legend.fontsize": 11,
})

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, width, height, text, fontsize=12, lw=1.7):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + width/2, y + height/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=1.7, mutation_scale=14, connectionstyle="arc3"):
    patch = FancyArrowPatch(
        start, end, arrowstyle="->", mutation_scale=mutation_scale,
        linewidth=lw, color="black", connectionstyle=connectionstyle
    )
    ax.add_patch(patch)
    return patch

def bidir(ax, start, end, lw=1.5, mutation_scale=13, connectionstyle="arc3"):
    patch = FancyArrowPatch(
        start, end, arrowstyle="<->", mutation_scale=mutation_scale,
        linewidth=lw, color="black", connectionstyle=connectionstyle
    )
    ax.add_patch(patch)
    return patch

## 1. Repository transition

Notebook 53 is the distributed-systems notebook in the second arc.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_title("Distributed scheduling continues the second systems arc", pad=24)

first = [("00\nContext",0.04),("07\nVerification\nResource",0.19),("13\nConfidence\nScheduling",0.34),
("17\nSemi-AR\nDrafting",0.49),("23\nThroughput\nObjective",0.64),("29\nHardware\nConstraints",0.79),("37\nOperating\nRegimes",0.91)]
w,h,y = 0.10,0.12,0.66
for label,x0 in first:
    rounded_box(ax,(x0,y),w,h,label,fontsize=10)
for (_,x1),(_,x2) in zip(first[:-1],first[1:]):
    arrow(ax,(x1+w,y+h/2),(x2,y+h/2),lw=1.4,mutation_scale=10)
ax.text(0.5,0.56,"first arc: mechanism → operating regime",ha="center",va="center",fontsize=13)
ax.plot([0.06,0.94],[0.51,0.51],color="black",lw=1.4)

second = [("43\nResource\nAllocation",0.28),("49\nAdaptive\nInfrastructure",0.46),
("53\nDistributed\nScheduling",0.64),("59\nCluster\nOptimization",0.82)]
w2,h2,y2=0.14,0.12,0.28
for label,x0 in second:
    rounded_box(ax,(x0,y2),w2,h2,label,fontsize=10)
for (_,x1),(_,x2) in zip(second[:-1],second[1:]):
    arrow(ax,(x1+w2,y2+h2/2),(x2,y2+h2/2),lw=1.4,mutation_scale=10)
ax.text(0.59,0.15,"second arc: controller → infrastructure → distributed scheduling → cluster optimization",ha="center",va="center",fontsize=13)
savefig("53_repository_transition.png")

## 2. Distributed serving map

A single adaptive infrastructure loop becomes a distributed system when each region or node has its own queue, router, replica pool, telemetry stream, and local controller.

In [ ]:
fig, ax = plt.subplots(figsize=(15,9))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Distributed adaptive serving map", pad=22)
regions = {"Region A":(0.08,0.58), "Region B":(0.60,0.58), "Region C":(0.34,0.18)}
for name,(x,y) in regions.items():
    rounded_box(ax,(x,y),0.28,0.26,name,fontsize=12,lw=1.9)
    rounded_box(ax,(x+0.02,y+0.16),0.10,0.06,"router",fontsize=9)
    rounded_box(ax,(x+0.15,y+0.16),0.10,0.06,"queue",fontsize=9)
    rounded_box(ax,(x+0.02,y+0.06),0.10,0.06,"replicas",fontsize=9)
    rounded_box(ax,(x+0.15,y+0.06),0.10,0.06,"telemetry",fontsize=9)
rounded_box(ax,(0.40,0.78),0.20,0.09,"shared telemetry\nfabric",fontsize=12)
rounded_box(ax,(0.40,0.47),0.20,0.09,"coordination\npolicy",fontsize=12)
for (x,y) in regions.values():
    arrow(ax,(x+0.20,y+0.26),(0.50,0.78),lw=1.2,mutation_scale=10)
    arrow(ax,(0.50,0.47),(x+0.14,y+0.13),lw=1.2,mutation_scale=10)
bidir(ax,(0.36,0.70),(0.60,0.70),connectionstyle="arc3,rad=0.18")
bidir(ax,(0.24,0.58),(0.42,0.44),connectionstyle="arc3,rad=-0.18")
bidir(ax,(0.68,0.58),(0.50,0.44),connectionstyle="arc3,rad=0.18")
ax.text(0.5,0.05,"Local serving loops coordinate through shared telemetry and bounded migration.",ha="center",fontsize=13)
savefig("53_distributed_serving_map.png")

## 3. Local versus global control

Local controllers handle fast decisions. Global coordination handles slower capacity and migration decisions.

In [ ]:
fig, ax = plt.subplots(figsize=(14,7))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Local controllers and global coordination act at different time scales", pad=20)
rounded_box(ax,(0.08,0.62),0.28,0.18,"local controller\nmilliseconds\nrouting, admission,\nreplica choice",fontsize=12)
rounded_box(ax,(0.64,0.62),0.28,0.18,"global coordinator\nseconds-minutes\nmigration, placement,\ncapacity balance",fontsize=12)
rounded_box(ax,(0.36,0.32),0.28,0.16,"shared telemetry\nqueue depth, utilization,\nlatency, memory pressure",fontsize=12)
arrow(ax,(0.36,0.71),(0.64,0.71))
arrow(ax,(0.64,0.65),(0.36,0.43))
arrow(ax,(0.22,0.62),(0.43,0.48))
arrow(ax,(0.50,0.48),(0.78,0.62))
bidir(ax,(0.36,0.68),(0.64,0.68),connectionstyle="arc3,rad=-0.25")
ax.text(0.50,0.14,"Fast local control prevents latency spikes; slower global control prevents persistent imbalance.",ha="center",fontsize=13)
savefig("53_local_vs_global_control.png")

## 4. Shared telemetry fabric

The telemetry fabric converts many local states into a shared cluster state.

In [ ]:
fig, ax = plt.subplots(figsize=(14,7.5))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Shared telemetry fabric aggregates local serving state", pad=20)
nodes=[(0.10,0.68),(0.10,0.40),(0.10,0.12),(0.74,0.68),(0.74,0.40),(0.74,0.12)]
for i,(x,y) in enumerate(nodes,1):
    rounded_box(ax,(x,y),0.16,0.10,f"node {i}\nq,u,m,λ",fontsize=10)
rounded_box(ax,(0.40,0.50),0.20,0.12,"telemetry\naggregator",fontsize=12)
rounded_box(ax,(0.40,0.28),0.20,0.12,"cluster\nstate S(t)",fontsize=12)
for x,y in nodes:
    arrow(ax,(x+0.16,y+0.05),(0.40,0.56),lw=1.1,mutation_scale=9)
arrow(ax,(0.50,0.50),(0.50,0.40))
for x,y in nodes:
    arrow(ax,(0.50,0.28),(x,y+0.03),lw=0.9,mutation_scale=8)
ax.text(0.5,0.06,r"$S(t)=\{s_i(t)\}_{i=1}^{N}$",ha="center",fontsize=18)
savefig("53_shared_telemetry_fabric.png")

## 5. Load migration surface

Migration is useful when local pressure is high, remote spare capacity is available, and coordination overhead remains acceptable.

In [ ]:
local_pressure=np.linspace(0.05,1.0,120); remote_spare=np.linspace(0.05,1.0,120)
P,R=np.meshgrid(local_pressure,remote_spare)
score=1.15*P+0.95*R-0.55*(P-R)**2
policy=np.zeros_like(score,dtype=int)
policy[(score>1.05)&(R>0.35)]=1
policy[(score>1.45)&(R>0.55)]=2
policy[(P>0.85)&(R<0.25)]=3
fig,ax=plt.subplots(figsize=(10.5,8))
im=ax.imshow(policy,origin="lower",extent=[local_pressure.min(),local_pressure.max(),remote_spare.min(),remote_spare.max()],aspect="auto")
cbar=plt.colorbar(im,ticks=[0,1,2,3]); cbar.ax.set_yticklabels(["keep local","partial migrate","migrate","shed/shorten"])
cbar.set_label("selected migration policy")
ax.set_title("Load migration surface"); ax.set_xlabel("local pressure"); ax.set_ylabel("remote spare capacity")
ax.text(0.25,0.25,"keep\nlocal",ha="center",fontsize=12)
ax.text(0.58,0.48,"partial\nmigrate",ha="center",fontsize=12)
ax.text(0.78,0.78,"migrate",ha="center",fontsize=12)
ax.text(0.90,0.16,"shed /\nshorten",ha="center",fontsize=12)
savefig("53_load_migration_surface.png")

## 6. Replica placement policy

Replica placement must account for heterogeneous nodes.

In [ ]:
nodes=["node A","node B","node C","node D","node E"]
roles=["draft","verify","mixed","reserve"]
placement=np.array([[0.55,0.20,0.20,0.05],[0.18,0.55,0.20,0.07],[0.30,0.30,0.32,0.08],[0.20,0.25,0.25,0.30],[0.42,0.18,0.25,0.15]])
fig,ax=plt.subplots(figsize=(12,6.5))
left=np.zeros(len(nodes))
for j,role in enumerate(roles):
    ax.barh(nodes,placement[:,j],left=left,label=role)
    for i,val in enumerate(placement[:,j]):
        if val>0.10:
            ax.text(left[i]+val/2,i,f"{role}\n{val:.2f}",ha="center",va="center",fontsize=9)
    left += placement[:,j]
ax.set_xlim(0,1); ax.set_xlabel("replica placement fraction")
ax.set_title("Replica placement policy across heterogeneous nodes")
ax.grid(axis="x",alpha=0.25)
ax.legend(ncol=4,loc="lower center",bbox_to_anchor=(0.5,-0.27))
savefig("53_replica_placement_policy.png")

## 7. Queue balancing dynamics

Distributed scheduling should reduce persistent imbalance, not merely react to it locally.

In [ ]:
t=np.arange(0,30)
initial=np.array([42,18,55,24,36],dtype=float)
target=initial.mean()
queues=[]; q=initial.copy()
for _ in t:
    queues.append(q.copy())
    q=q+0.18*(target-q)+np.array([0.5,-0.1,-0.4,0.2,-0.2])
queues=np.array(queues)
fig,ax=plt.subplots(figsize=(12,6.5))
for i in range(queues.shape[1]):
    ax.plot(t,queues[:,i],marker="o",markevery=5,label=f"node {i+1}")
ax.axhline(target,linestyle="--",color="black",label="cluster mean target")
ax.set_title("Queue balancing dynamics across distributed nodes")
ax.set_xlabel("coordination step"); ax.set_ylabel("queue depth")
ax.grid(alpha=0.3); ax.legend(ncol=3)
savefig("53_queue_balancing_dynamics.png")

## 8. Distributed routing graph

Schedulers are vertices and migration/routing options are edges.

In [ ]:
fig,ax=plt.subplots(figsize=(10,8))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Distributed routing graph",pad=20)
pos={"A":(0.18,0.70),"B":(0.50,0.82),"C":(0.82,0.70),"D":(0.28,0.28),"E":(0.72,0.28)}
edges=[("A","B"),("B","C"),("A","D"),("C","E"),("D","E"),("B","D"),("B","E")]
for u,v in edges:
    bidir(ax,pos[u],pos[v],lw=1.3,mutation_scale=11)
for name,(x,y) in pos.items():
    circ=Circle((x,y),0.055,edgecolor="black",facecolor="white",lw=1.8)
    ax.add_patch(circ)
    ax.text(x,y,name,ha="center",va="center",fontsize=14,weight="bold")
    ax.text(x,y-0.085,"scheduler",ha="center",va="center",fontsize=9)
ax.text(0.5,0.08,"Edges represent admissible migration or replica-routing paths.",ha="center",fontsize=13)
savefig("53_distributed_routing_graph.png")

## 9. Failure reroute path

Distributed scheduling should preserve service when one node overloads or fails.

In [ ]:
fig,ax=plt.subplots(figsize=(14,6))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Distributed failure reroute path",pad=20)
main=[("node\nhealthy",0.06),("overload /\nfailure",0.24),("telemetry\nalert",0.43),("reroute\ntraffic",0.63),("recover /\nrebalance",0.82)]
w,h,y=0.13,0.14,0.55
for i,(label,x) in enumerate(main):
    rounded_box(ax,(x,y),w,h,label,fontsize=12)
    if i<len(main)-1:
        arrow(ax,(x+w,y+h/2),(main[i+1][1],y+h/2))
rounded_box(ax,(0.41,0.22),0.18,0.12,"fallback:\nlocal shorten",fontsize=12)
rounded_box(ax,(0.65,0.22),0.18,0.12,"global:\ncapacity shift",fontsize=12)
arrow(ax,(0.49,0.55),(0.50,0.34)); arrow(ax,(0.59,0.28),(0.65,0.28)); arrow(ax,(0.74,0.34),(0.70,0.55),connectionstyle="arc3,rad=-0.18")
ax.text(0.5,0.10,"Distributed fallback keeps local failures from becoming cluster-wide failures.",ha="center",fontsize=13)
savefig("53_failure_reroute_path.png")

## 10. Coordination overhead tradeoff

Coordination has value, but it is not free.

In [ ]:
coord=np.linspace(0,1,100)
benefit=1.25*(1-np.exp(-4.0*coord))
overhead=0.18+0.95*coord**2
net=benefit-overhead
fig,ax=plt.subplots(figsize=(12,6.5))
ax.plot(coord,benefit,label="coordination benefit",linewidth=2)
ax.plot(coord,overhead,label="coordination overhead",linewidth=2)
ax.plot(coord,net,label="net gain",linewidth=2)
best=coord[np.argmax(net)]
ax.axvline(best,linestyle="--",color="black",label=f"best ≈ {best:.2f}")
ax.set_title("Coordination overhead tradeoff")
ax.set_xlabel("coordination intensity"); ax.set_ylabel("normalized value")
ax.grid(alpha=0.3); ax.legend()
savefig("53_coordination_overhead_tradeoff.png")

## 11. Final synthesis

Notebook 53 turns adaptive infrastructure into coordinated distributed serving.

In [ ]:
fig,ax=plt.subplots(figsize=(15,5.8))
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
ax.set_title("Distributed scheduling specifies coordinated adaptive serving",pad=20)
labels=["local\ncontrollers","shared\ntelemetry","distributed\nqueues","replica\nplacement","load\nmigration","coordinated\nserving"]
xs=np.linspace(0.05,0.82,len(labels)); w,h,y=0.13,0.18,0.55
for x,label in zip(xs,labels):
    rounded_box(ax,(x,y),w,h,label,fontsize=12)
for x1,x2 in zip(xs[:-1],xs[1:]):
    arrow(ax,(x1+w,y+h/2),(x2,y+h/2))
rounded_box(ax,(0.25,0.20),0.50,0.11,"Local adaptive infrastructure becomes coordinated distributed serving.",fontsize=12)
savefig("53_final_synthesis.png")

## Key equations

Local serving state:

\[
s_i(t)=(q_i(t),u_i(t),m_i(t),\lambda_i(t)).
\]

Cluster state:

\[
S(t)=\{s_i(t)\}_{i=1}^{N}.
\]

Routing or migration decision:

\[
r_{ij}(t)=R(s_i(t),s_j(t),x_t).
\]

Local allocation under global state:

\[
a_i(t)=A(s_i(t),S(t)).
\]

Queue update with migration:

\[
q_i(t+1)
=
h_i\left(q_i(t),a_i(t),\sum_j r_{ji}(t)-\sum_j r_{ij}(t)\right).
\]

Coordination objective:

\[
\max_R
\sum_i \Theta_i(a_i)
-
\Omega(R).
\]

## Notebook summary

Notebook 53 specifies coordinated adaptive serving:

- local controllers keep serving loops responsive;
- shared telemetry exposes cluster state;
- routing and migration distribute pressure;
- placement policies account for heterogeneous nodes;
- failure reroute paths prevent local overload from becoming global failure;
- coordination improves balance until overhead dominates.

Next notebook: **59 — Cluster Optimization**.

## Download notebook artifacts

Run the final cell to package Notebook 53 outputs for download.

The archive includes generated `53_*.png` figures, manifest JSON, and the notebook itself when available.

In [ ]:
manifest = {
    "notebook": "53_distributed_scheduling.ipynb",
    "title": "Distributed Scheduling",
    "engineering_statement": "Distributed scheduling specifies coordinated adaptive serving.",
    "figures": sorted([p.name for p in FIGURES.glob("53_*.png")]),
    "previous_notebook": "49_adaptive_infrastructure.ipynb",
    "next_notebook": "59_cluster_optimization.ipynb",
}
manifest_path = RESULTS / "53_distributed_scheduling_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = RESULTS / "53_distributed_scheduling_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(FIGURES.glob("53_*.png")):
        zf.write(p, arcname=f"figures/{p.name}")
    for p in sorted(RESULTS.glob("*")):
        if p.is_file() and p.resolve() != zip_path.resolve():
            zf.write(p, arcname=f"results/53_distributed_scheduling/{p.name}")
    candidates = [
        Path.cwd() / "53_distributed_scheduling.ipynb",
        REPO_ROOT / "notebooks" / "53_distributed_scheduling.ipynb",
    ]
    for nb_path in candidates:
        if nb_path.exists():
            zf.write(nb_path, arcname=f"notebooks/{nb_path.name}")
            break

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass